# Program Generator Notebook

This notebook is fully standalone: all prompt-building, data loading, retry logic, and code generation helpers live inside the notebook itself.

It reads attention files from `data/attention`, writes generated programs into `data/<strategy>/generated_code`, and only depends on the notebook kernel plus the Anthropic API client.

Before running generation, make sure your Anthropic API credentials are available in the environment.


In [8]:
import importlib.util
import subprocess
import sys

required_packages = ["anthropic", "torch", "transformers"]
missing_packages = [pkg for pkg in required_packages if importlib.util.find_spec(pkg) is None]
if missing_packages:
    subprocess.check_call([sys.executable, "-m", "pip", "install", *missing_packages])


In [ ]:
import ast
import hashlib
import json
import random
import re
import time
from pathlib import Path
from typing import Any

import anthropic
import numpy as np
import torch
from transformers import GPT2LMHeadModel, GPT2Tokenizer


PROJECT_ROOT = Path.cwd()
DATA_DIR = PROJECT_ROOT / "data"
CACHE_DIR = PROJECT_ROOT / ".notebook_cache"
ATTENTION_CACHE_DIR = CACHE_DIR / "attention"
MODEL_NAME = "gpt2"
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"

NUM_LAYERS = 12
NUM_HEADS = 12
NUM_EXAMPLE_SENTENCES = 5
SEED = 42
MIN_ATTENTION_WEIGHT = 0.01
MAX_BORING_PER_STRATUM = 3
STRATA_QUANTILES = [
    ("high", 0.75, 1.0, 15),
    ("upper-mid", 0.50, 0.75, 10),
    ("lower-mid", 0.25, 0.50, 10),
    ("low", 0.0, 0.25, 10),
]

CLAUDE_MODEL = "claude-sonnet-4-20250514"
MAX_RETRIES = 3
RETRY_BASE_DELAY = 5
SYSTEM_PROMPT = (
    "You are an expert at analyzing transformer attention patterns and writing "
    "Python code to approximate them. You write clean, correct Python that uses "
    "spacy and numpy to predict attention patterns based on linguistic features."
)

EXAMPLE_SENTENCES = [
    "The small cat sat on the warm window sill.",
    "After the rain stopped, children ran into the park.",
    "A musician who studied in Paris played softly.",
    "The red book that you borrowed is on the table.",
    "My friend and I walked home because the train was late.",
    "When the phone rang, she answered immediately.",
    "The chef prepared soup while the guests waited in silence.",
    "Although the road was icy, the driver continued carefully.",
    "Several bright stars appeared above the dark hill.",
    "The scientist explained the result with a simple diagram.",
    "The child who lost the toy began to cry.",
    "Before lunch, we finished the last exercise.",
    "A tall tree blocked the view from the balcony.",
    "The artist painted the wall with blue and gold stripes.",
    "Because the meeting ended early, everyone left before sunset.",
    "The old machine in the corner still worked surprisingly well.",
]

_TOKENIZER = None
_MODEL = None


def strategy_dirs(name: str) -> tuple[Path, Path, Path]:
    base = DATA_DIR / name
    return base / "generated_code", base / "predictions", base / "results"


def load_attention(identifier) -> dict[str, Any]:
    """Load an attention cache entry.

    `identifier` may be:
    - int: interpreted as an index; tries sentence_{idx:04d}.npz, then the nth file in the cache directory
    - str or Path: treated as a filename (with or without .npz) inside ATTENTION_CACHE_DIR
    """
    ATTENTION_CACHE_DIR.mkdir(parents=True, exist_ok=True)

    # Resolve path based on identifier type
    path = None
    if isinstance(identifier, (int, np.integer)):
        path1 = ATTENTION_CACHE_DIR / f"sentence_{int(identifier):04d}.npz"
        if path1.exists():
            path = path1
        else:
            files = sorted(ATTENTION_CACHE_DIR.glob("*.npz"))
            if len(files) > int(identifier):
                path = files[int(identifier)]
    else:
        p = Path(identifier)
        if p.exists():
            path = p
        else:
            p2 = ATTENTION_CACHE_DIR / f"{str(identifier)}.npz"
            if p2.exists():
                path = p2

    if path is None or not Path(path).exists():
        raise FileNotFoundError(f"No attention cache found for {identifier} in {ATTENTION_CACHE_DIR}")

    data = np.load(path, allow_pickle=True)
    return {
        "tokens": list(data["tokens"]),
        "sentence": str(data["sentence"]),
        "attention": data["attention"],
    }


def _attention_cache_path(sentence: str) -> Path:
    ATTENTION_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    digest = hashlib.sha256(sentence.encode("utf-8")).hexdigest()[:16]
    return ATTENTION_CACHE_DIR / f"{digest}.npz"


def get_model_and_tokenizer():
    global _TOKENIZER, _MODEL
    if _TOKENIZER is None:
        _TOKENIZER = GPT2Tokenizer.from_pretrained(MODEL_NAME)
    if _MODEL is None:
        _MODEL = GPT2LMHeadModel.from_pretrained(
            MODEL_NAME,
            attn_implementation="eager",
        )
        _MODEL.config.output_attentions = True
        _MODEL.eval()
        _MODEL.to(DEVICE)
    return _TOKENIZER, _MODEL


def extract_attention_example(sentence: str) -> dict[str, Any]:
    """Compute and cache attention for a single sentence. Returns a dict with tokens, sentence, attention array."""
    ATTENTION_CACHE_DIR.mkdir(parents=True, exist_ok=True)
    cache_path = _attention_cache_path(sentence)
    if cache_path.exists():
        data = np.load(cache_path, allow_pickle=True)
        return {
            "tokens": list(data["tokens"]),
            "sentence": str(data["sentence"]),
            "attention": data["attention"],
        }

    tokenizer, model = get_model_and_tokenizer()
    input_ids = tokenizer.encode(sentence, return_tensors="pt")
    # Use convert_ids_to_tokens for stable, BPE-level tokens
    tokens = tokenizer.convert_ids_to_tokens(input_ids[0])
    input_ids = input_ids.to(DEVICE)

    with torch.no_grad():
        outputs = model(input_ids, output_attentions=True)

    attention = np.stack(
        [layer_attn[0].detach().cpu().numpy() for layer_attn in outputs.attentions],
        axis=0,
    )

    # Cache compressed
    np.savez_compressed(
        cache_path,
        tokens=tokens,
        sentence=sentence,
        attention=attention,
    )
    return {
        "tokens": tokens,
        "sentence": sentence,
        "attention": attention,
    }


def load_train_data(limit: int | None = None) -> list[dict[str, Any]]:
    """Load training attention examples.

    If `data/attention/index.json` exists, it is expected to contain a dict with a
    "train" key listing identifiers (ints or filenames). Otherwise falls back to
    `EXAMPLE_SENTENCES` and will compute & cache attentions.
    """
    local_index = DATA_DIR / "attention" / "index.json"
    if local_index.exists():
        with open(local_index, encoding="utf-8") as f:
            split = json.load(f)
        train_indices = split.get("train", [])
        if limit is not None:
            train_indices = train_indices[:limit]
        return [load_attention(i) for i in train_indices]

    sentences = EXAMPLE_SENTENCES if limit is None else EXAMPLE_SENTENCES[:limit]
    return [extract_attention_example(sentence) for sentence in sentences]

# The remainder of the cell (represent_stratified_pairs, format_head_examples, build_prompt,
# call_claude, extract_code, validate_code, generate_with_retries, generate_program_for_head,
# generate_programs, etc.) remains unchanged from the original notebook and depends on these
# corrected helpers.  These functions rely on the stable tokenization and cache semantics above.
print("Core attention helpers loaded — cache dir:", ATTENTION_CACHE_DIR)


In [ ]:
# Consolidation cell — avoid duplicate/conflicting definitions
# Ensure a single ATTENTION_DIR is available and a robust load_train_data is exposed.
ATTENTION_DIR = DATA_DIR / "attention"
ATTENTION_DIR.mkdir(parents=True, exist_ok=True)

# Provide a convenience loader that prefers an index.json if present,
# otherwise falls back to the main `load_train_data(limit=None)` implementation above.

def load_train_data_from_index(limit: int | None = None) -> list[dict]:
    idx_file = ATTENTION_DIR / "index.json"
    if idx_file.exists():
        with open(idx_file, encoding="utf-8") as f:
            split = json.load(f)
        indices = split.get("train", [])
        if limit is not None:
            indices = indices[:limit]
        return [load_attention(i) for i in indices]
    # fallback to canonical loader
    return load_train_data(limit=limit)

# Expose the canonical name used elsewhere in the notebook
load_train_data = load_train_data_from_index

print("ATTENTION_DIR set to:", ATTENTION_DIR)
print("load_train_data() now uses index.json if present, else computes/cache examples.")


In [10]:
# Configure what to generate here.
strategy = "standalone_zero_shot"
model = None
layer = 0        # Change to None to generate all layers.
head = 0         # Change to None to generate a full layer.
overwrite = False
run_now = False  # Set to True when you want to launch generation.

generated_paths: list[Path] = []
if run_now:
    generated_paths = generate_programs(
        layer=layer,
        head=head,
        strategy=strategy,
        model=model,
        overwrite=overwrite,
    )
    print(f"Generated {len(generated_paths)} file(s).")
else:
    print("Set run_now = True in the configuration cell to generate programs.")


Set run_now = True in the configuration cell to generate programs.


In [11]:
def preview_program(path: Path, max_lines: int = 60) -> str:
    lines = path.read_text(encoding="utf-8").splitlines()
    return "\n".join(lines[:max_lines])


if generated_paths:
    print(preview_program(generated_paths[0]))
